# Sketch2Vision Pix2Pix demo — CMP Facades

**Demonstration only.** This trains the existing Sketch2Vision U-Net Generator and conditional PatchGAN Discriminator on a small subset of paired architectural label-map/photo data. It is not the intended Sketch2Vision dataset and does not demonstrate freehand-sketch understanding. The input used here is a Canny outline derived from the dataset's facade label map.

## Dataset terms checkpoint (read before running the download cell)
The official [CMP Facade Database page](https://cmp.felk.cvut.cz/~tylecr1/facade/) describes the work as **CC BY-SA**, but the visible page does not specify a version and says images came from various sources. Attribution is required and ShareAlike conditions apply under the applicable license; underlying image-level rights and downstream model implications remain unresolved here. The [Pix2Pix Berkeley archive](https://efrosgans.eecs.berkeley.edu/pix2pix/datasets/) is a convenient prepared copy, not a separate license grant. This notebook does **not** approve the dataset for Sketch2Vision, commercial use, redistribution, or final training. Review the linked terms and project requirements before downloading or running.

The production `train.py` provenance/license gate is unchanged. This separate notebook calls the existing low-level GAN training step only for a visibly isolated teaching/demo run; its checkpoints are marked `demo_only` and must not be moved into the project checkpoints or treated as a production model.

In [ ]:
# In Colab: Runtime > Change runtime type > select a GPU (T4 if offered).
# This cell uploads a current project source ZIP and extracts it under /content/Sketch2Vision.
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import stat

PROJECT_ROOT = Path('/content/Sketch2Vision')
if not PROJECT_ROOT.exists():
    uploaded = files.upload()  # Select Sketch2Vision-Facades-Demo-Colab.zip
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise RuntimeError('Upload exactly one current Sketch2Vision source ZIP.')
    with ZipFile(zip_names[0]) as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'Unsafe ZIP member: {info.filename}')
        archive.extractall('/content')
if not (PROJECT_ROOT / 'backend/ml/facades_demo.py').is_file():
    raise FileNotFoundError('This source ZIP is missing backend/ml/facades_demo.py; create/upload a fresh ZIP from the updated project.')
import sys
sys.path.insert(0, str(PROJECT_ROOT))
print('Project source:', PROJECT_ROOT)

In [ ]:
# Set True only after you personally review the dataset terms in the markdown above.
DEMO_TERMS_REVIEWED = False
assert DEMO_TERMS_REVIEWED, 'Review CMP Facades CC BY-SA terms and provenance before the demo download.'

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training; no CPU fallback is enabled for this presentation run.')
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
import random, numpy as np
from torch.utils.data import DataLoader
from backend.ml.facades_demo import (
    download_facades, extract_facades, FacadesPairDataset, deterministic_subset,
    train_facades_demo, predict_one, tensor_to_image,
)
from backend.ml.models import build_pix2pix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DATA_DIR = Path('/content/facades-demo-data')
archive_path = download_facades(DATA_DIR)  # ~29 MB official Berkeley Pix2Pix archive
dataset_root = extract_facades(archive_path, DATA_DIR / 'extracted')
print('Dataset folder:', dataset_root)
for split in ('train', 'val', 'validation', 'test'):
    try:
        ds = FacadesPairDataset(dataset_root, split, 128)
        print(split, len(ds), 'paired examples')
    except FileNotFoundError:
        pass

IMAGE_SIZE = 128
BATCH_SIZE = 1
MAX_TRAIN = 160  # deterministic quick-demo subset; set None for full split
MAX_VALIDATION = 32
EPOCHS = 3
BASE_CHANNELS = 16  # same U-Net/PatchGAN, narrower than project baseline 32
train_ds = deterministic_subset(FacadesPairDataset(dataset_root, 'train', IMAGE_SIZE), MAX_TRAIN, SEED)
try:
    val_base = FacadesPairDataset(dataset_root, 'val', IMAGE_SIZE)
except FileNotFoundError:
    val_base = FacadesPairDataset(dataset_root, 'validation', IMAGE_SIZE)
val_ds = deterministic_subset(val_base, MAX_VALIDATION, SEED + 1)
test_ds = FacadesPairDataset(dataset_root, 'test', IMAGE_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
print(f'Demo subset: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}; these are not project dataset counts.')

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

# Visual check: paired edge conditioning and target photo, before training.
sample = test_ds[0]
plt.figure(figsize=(7, 3))
plt.subplot(1, 2, 1); plt.imshow(tensor_to_image(sample['edges']), cmap='gray'); plt.title('Input: label-map outline'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(tensor_to_image(sample['targets'])); plt.title('Paired facade photo'); plt.axis('off')
plt.tight_layout(); plt.show()

generator, discriminator = build_pix2pix(base_channels=BASE_CHANNELS, num_downs=7)
initial = predict_one(generator, sample['edges'], device)
plt.figure(figsize=(10, 3))
for i, (img, title, cmap) in enumerate([(tensor_to_image(sample['edges']), 'Input outline', 'gray'), (tensor_to_image(initial), 'Generator at initialization', None), (tensor_to_image(sample['targets']), 'Target photo', None)], 1):
    plt.subplot(1, 3, i); plt.imshow(img, cmap=cmap); plt.title(title); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
OUTPUT_DIR = Path('/content/sketch2vision-facades-demo-run')
fixed_input = sample['edges']
fixed_target = sample['targets']

def show_epoch(epoch, metrics):
    generated = predict_one(generator, fixed_input, device)
    print(f"Epoch {epoch}: G={metrics['generator_total']:.4f}, D={metrics['discriminator']:.4f}, validation L1={metrics['validation_l1']:.4f}")
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    for ax, image, title, cmap in zip(axes, [tensor_to_image(fixed_input), tensor_to_image(generated), tensor_to_image(fixed_target)], ['Input outline', 'Generated after epoch', 'Target photo'], ['gray', None, None]):
        ax.imshow(image, cmap=cmap); ax.set_title(title); ax.axis('off')
    plt.show()

history = train_facades_demo(
    generator, discriminator, train_loader, val_loader, device, OUTPUT_DIR,
    epochs=EPOCHS, image_size=IMAGE_SIZE, learning_rate=2e-4, l1_weight=100.0, callback=show_epoch,
)
print('Demo-only checkpoint:', OUTPUT_DIR / 'best.pt')

In [ ]:
# Loss curves from the actual training run.
epochs = [row['epoch'] for row in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [row['generator_total'] for row in history], marker='o', label='Generator total')
plt.plot(epochs, [row['discriminator'] for row in history], marker='o', label='Discriminator')
plt.plot(epochs, [row['validation_l1'] for row in history], marker='o', label='Validation L1')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Facades demonstration run'); plt.grid(True, alpha=.25); plt.legend(); plt.show()

# Inference on a held-out Facades input. This is a dataset input outline, not a freehand sketch.
final_output = predict_one(generator, test_ds[0]['edges'], device)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, image, title, cmap in zip(axes, [tensor_to_image(test_ds[0]['edges']), tensor_to_image(final_output), tensor_to_image(test_ds[0]['targets'])], ['Input outline', 'Pix2Pix output', 'Paired target'], ['gray', None, None]):
    ax.imshow(image, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.show()

from google.colab import files
files.download(str(OUTPUT_DIR / 'best.pt'))  # demo checkpoint; not a Sketch2Vision production checkpoint

## What this run proves—and does not prove
The notebook runs the project's actual U-Net Generator, PatchGAN Discriminator, adversarial + weighted L1 losses, optimizer updates, validation metric, and checkpoint writing. A few epochs on a small subset may produce weak or blurry results; that is still a genuine training run if the cells complete and the saved checkpoint is written.

It does not train on the intended 10-category image set, establish image rights for the Sketch2Vision dataset, validate the model for broad sketch-to-photo generation, or prove real freehand-sketch generalization. The facade outline comes from a semantic label map.